In [3]:
# ============================================================
# U-Net Training — Combined Input (nM with channel 0 := nM11s)
# Fixes:
#   - Eliminate DataLoader spawn/pickle crash on macOS/MPS.
#   - Auto-select device: CUDA > MPS > CPU.
#   - Force single-process loading on MPS/notebook to avoid
#     "Can't get attribute 'CombinedH5Dataset' on __main__".
#   - Disable pin_memory on MPS.
# ============================================================

import os
import sys
import json
import math
import time
import platform
from pathlib import Path
from typing import Optional, Tuple, Dict

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# -------------------------
# Device + Dataloader policy
# -------------------------
def resolve_device(preferred: str = 'cuda') -> torch.device:
    if preferred.startswith('cuda') and torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

def dataloader_runtime_policy(device: torch.device,
                              requested_workers: int,
                              requested_pin_memory: bool) -> Dict:
    # macOS + MPS + notebooks have fragile spawn; force workers=0
    is_macos = (platform.system() == 'Darwin')
    in_ipython = ('ipykernel' in sys.modules) or ('IPython' in sys.modules)
    use_workers = requested_workers

    if device.type == 'mps' or is_macos and in_ipython:
        use_workers = 0

    # pin_memory only makes sense for CUDA
    pin_mem = requested_pin_memory and (device.type == 'cuda')
    return {
        'num_workers': use_workers,
        'pin_memory': pin_mem,
        'persistent_workers': (use_workers > 0)
    }

# -------------------------
# Model
# -------------------------
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        dy = x2.size(2) - x1.size(2)
        dx = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True):
        super().__init__()
        self.bilinear = bilinear
        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1   = Up(1024, 512 // factor, bilinear)
        self.up2   = Up(512, 256 // factor, bilinear)
        self.up3   = Up(256, 128 // factor, bilinear)
        self.up4   = Up(128, 64, bilinear)
        self.outc  = nn.Conv2d(64, n_classes, kernel_size=1)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x  = self.up1(x5, x4)
        x  = self.up2(x, x3)
        x  = self.up3(x, x2)
        x  = self.up4(x, x1)
        return self.outc(x)

# -------------------------
# Dataset (top-level class; pickle-safe)
# -------------------------
class CombinedH5Dataset(Dataset):
    """
    HDF5 combined dataset with inputs (H,W,16) and targets (H,W).
    Resizes to target_size. Optional flips.
    """
    def __init__(self, h5_path: Path, target_size: Tuple[int,int]=(512,512), augment: bool=False):
        super().__init__()
        self.h5_path = str(h5_path)
        self.target_size = target_size
        self.augment = augment
        with h5py.File(self.h5_path, 'r') as f:
            self.num_samples = int(f.attrs['num_samples'])
            self.num_classes = int(f.attrs['num_classes'])
            self.class_labels = f['class_labels'][:] if 'class_labels' in f else None

    def __len__(self):
        return self.num_samples

    def _random_flip(self, x: torch.Tensor, y: torch.Tensor):
        if torch.rand(1).item() < 0.5:
            x = torch.flip(x, dims=[2])
            y = torch.flip(y, dims=[1])
        if torch.rand(1).item() < 0.5:
            x = torch.flip(x, dims=[1])
            y = torch.flip(y, dims=[0])
        return x, y

    def __getitem__(self, idx: int):
        with h5py.File(self.h5_path, 'r') as f:
            g = f['samples'][f'sample_{idx:06d}']
            inp = g['input'][:].astype(np.float32)   # (H,W,16)
            tgt = g['target'][:].astype(np.int64)    # (H,W)

        x = torch.from_numpy(inp).permute(2,0,1)     # (C,H,W)
        y = torch.from_numpy(tgt)                    # (H,W)

        x = F.interpolate(x.unsqueeze(0), size=self.target_size, mode='bilinear', align_corners=True).squeeze(0)
        y = F.interpolate(y.unsqueeze(0).unsqueeze(0).float(), size=self.target_size, mode='nearest').squeeze(0).squeeze(0).long()

        if self.augment:
            x, y = self._random_flip(x, y)

        return x, y

# -------------------------
# Class weights from HDF5
# -------------------------
def compute_class_weights_from_h5(h5_path: Path, smoothing: float=0.0) -> Optional[torch.Tensor]:
    try:
        with h5py.File(h5_path, 'r') as f:
            if 'target_class_distribution' not in f.attrs or 'class_labels' not in f:
                return None
            dist_json = f.attrs['target_class_distribution']
            if isinstance(dist_json, bytes):
                dist_json = dist_json.decode('utf-8')
            counts = json.loads(dist_json)
            labels = f['class_labels'][:].tolist()
        totals = np.array([counts.get(int(c), 0) for c in labels], dtype=np.float64)
        totals = totals + smoothing
        totals[totals <= 0] = 1.0
        inv_freq = 1.0 / totals
        weights = inv_freq / inv_freq.sum() * len(inv_freq)
        return torch.tensor(weights, dtype=torch.float32)
    except Exception:
        return None

# -------------------------
# Train / Validate
# -------------------------
def train_one_epoch(model, loader, optimizer, device, criterion, scaler=None):
    model.train()
    running_loss = 0.0
    n = 0
    for x, y in loader:
        x = x.to(device, non_blocking=False)
        y = y.to(device, non_blocking=False)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None and device.type == 'cuda':
            with torch.cuda.amp.autocast():
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
        running_loss += loss.item() * x.size(0)
        n += x.size(0)
    return running_loss / max(n,1)

@torch.no_grad()
def validate_epoch(model, loader, device, criterion):
    model.eval()
    loss_sum = 0.0
    n = 0
    correct = 0
    total = 0
    for x, y in loader:
        x = x.to(device, non_blocking=False)
        y = y.to(device, non_blocking=False)
        logits = model(x)
        loss = criterion(logits, y)
        loss_sum += loss.item() * x.size(0)
        n += x.size(0)
        pred = torch.argmax(logits, dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()
    avg_loss = loss_sum / max(n,1)
    acc = correct / max(total,1)
    return avg_loss, acc

# -------------------------
# Orchestrator
# -------------------------
def train_unet_model(
    train_h5_path: Path,
    val_h5_path: Path,
    target_size: Tuple[int,int]=(512,512),
    batch_size: int=4,
    num_epochs: int=20,
    learning_rate: float=1e-4,
    device: str='cuda',
    save_dir: str='./models/mueller_unet',
    bilinear: bool=True,
    use_amp: bool=True,
    num_workers: int=4,
):
    dev = resolve_device(device)
    os.makedirs(save_dir, exist_ok=True)

    # Build datasets
    ds_train = CombinedH5Dataset(train_h5_path, target_size=target_size, augment=True)
    ds_val   = CombinedH5Dataset(val_h5_path,   target_size=target_size, augment=False)

    # DataLoader policy
    dl_policy = dataloader_runtime_policy(dev, num_workers, requested_pin_memory=True)

    # DataLoaders (safe on macOS/MPS)
    train_loader = DataLoader(
        ds_train, batch_size=batch_size, shuffle=True,
        num_workers=dl_policy['num_workers'], pin_memory=dl_policy['pin_memory'],
        persistent_workers=dl_policy['persistent_workers'], drop_last=False
    )
    val_loader = DataLoader(
        ds_val, batch_size=batch_size, shuffle=False,
        num_workers=dl_policy['num_workers'], pin_memory=dl_policy['pin_memory'],
        persistent_workers=dl_policy['persistent_workers'], drop_last=False
    )

    n_channels = 16
    n_classes  = ds_train.num_classes

    model = UNet(n_channels=n_channels, n_classes=n_classes, bilinear=bilinear).to(dev)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

    class_weights = compute_class_weights_from_h5(train_h5_path)
    if class_weights is not None:
        class_weights = class_weights.to(dev)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # AMP only on CUDA
    scaler = torch.cuda.amp.GradScaler() if (use_amp and dev.type == 'cuda') else None

    best_val_loss = math.inf
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(1, num_epochs+1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, dev, criterion, scaler)
        val_loss, val_acc = validate_epoch(model, val_loader, dev, criterion)
        dt = time.time() - t0

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch:03d}/{num_epochs}  "
              f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  "
              f"time={dt:.1f}s  dev={dev.type}  workers={dl_policy['num_workers']}")

        ckpt = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_accuracy': val_acc,
            'config': {
                'n_channels': n_channels,
                'n_classes': n_classes,
                'bilinear': bilinear,
                'target_size': target_size,
            },
            'history': history
        }
        torch.save(ckpt, os.path.join(save_dir, 'last_model.pth'))
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(ckpt, os.path.join(save_dir, 'best_model.pth'))

    return model, history

# -------------------------
# ONNX Export
# -------------------------
from datetime import datetime

def export_to_onnx(
    model_path: str,
    onnx_path: str,
    target_size=(512, 512),
    opset_version: int = 18,            # do not request <18; avoid converter errors
    dynamic_batch: bool = True,
    use_dynamo: bool = True
):
    ckpt = torch.load(model_path, map_location='cpu')
    cfg = ckpt.get('config', {})
    n_channels = int(cfg.get('n_channels', 16))
    n_classes  = int(cfg.get('n_classes', 2))
    bilinear   = bool(cfg.get('bilinear', True))

    model = UNet(n_channels=n_channels, n_classes=n_classes, bilinear=bilinear)
    state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt['state_dict']
    model.load_state_dict(state_dict, strict=False)
    model.eval()

    # Dummy input
    dummy = torch.randn(1, n_channels, target_size[0], target_size[1])

    # Preferred path: dynamo_export with dynamic_shapes (no dynamic_axes)
    if use_dynamo:
        try:
            from torch.onnx import dynamo_export
            if dynamic_batch:
                # Define symbolic batch dim
                from torch.export import Dim
                batch = Dim("batch")
                exp = dynamo_export(
                    model,
                    dummy,
                    dynamic_shapes={"input": {0: batch}},   # only batch is dynamic
                )
            else:
                exp = dynamo_export(model, dummy)

            exp.save(onnx_path, opset_version=opset_version)
            # Persist metadata
            meta = {
                "n_channels": n_channels,
                "n_classes": n_classes,
                "target_size": list(target_size),
                "bilinear": bilinear,
                "opset": opset_version,
                "dynamic_batch": bool(dynamic_batch),
                "export_api": "torch.onnx.dynamo_export",
                "created": datetime.now().isoformat()
            }
            with open(Path(onnx_path).with_suffix('.json'), 'w') as f:
                json.dump(meta, f, indent=2)
            print(f"ONNX saved: {onnx_path} (dynamo, opset {opset_version})")
            return
        except Exception as e:
            print(f"[WARN] dynamo_export failed: {e}. Falling back to classic exporter.")

    # Fallback: classic exporter. Avoid dynamic_axes if torch sets dynamo=True internally.
    # Use fixed batch=1 to eliminate dynamic_axes entirely.
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        export_params=True,
        opset_version=opset_version,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes=None  # fixed shapes -> no dynamic_axes warning
    )
    meta = {
        "n_channels": n_channels,
        "n_classes": n_classes,
        "target_size": list(target_size),
        "bilinear": bilinear,
        "opset": opset_version,
        "dynamic_batch": False,
        "export_api": "torch.onnx.export",
        "created": datetime.now().isoformat()
    }
    with open(Path(onnx_path).with_suffix('.json'), 'w') as f:
        json.dump(meta, f, indent=2)
    print(f"ONNX saved: {onnx_path} (classic, opset {opset_version})")


# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    # Avoid accidental fork bombs on macOS notebooks
    if platform.system() == 'Darwin':
        try:
            import multiprocessing as mp
            mp.set_start_method('spawn', force=True)
        except RuntimeError:
            pass

    HDF5_DIR = Path("/Volumes/ep_ssd/MPL_Data/mmNoTissueFilter/hdf5_datasets/hdf5/combined")

    print("="*80)
    print("TRAINING — COMBINED (nM with c0 := nM11s or normalized M11s)")
    print("="*80)

    model, history = train_unet_model(
        train_h5_path=HDF5_DIR / "train.h5",
        val_h5_path=HDF5_DIR / "validation.h5",
        target_size=(512, 512),
        batch_size=4,
        num_epochs=100,
        learning_rate=1e-4,
        device='cuda',          # auto-fallback to mps/cpu inside
        save_dir='./models/mueller_unet',
        bilinear=True,
        use_amp=True,
        num_workers=4           # auto-forced to 0 on macOS/MPS/notebook
    )

    export_to_onnx(
        model_path='./models/mueller_unet/best_model.pth',
        onnx_path='./models/mueller_unet/mueller_unet.onnx',
        target_size=(512, 512),
        opset_version=12
    )

TRAINING — COMBINED (nM with c0 := nM11s or normalized M11s)


KeyError: "Unable to synchronously open attribute (can't locate attribute: 'num_samples')"